In [1]:
import os
import re
import random
from pathlib import Path
import json
import pandas as pd

In [2]:
# working directory for relative paths
notebooks_path = Path.cwd() # <repo-root>/noteboooks
repo_root_path = notebooks_path.parent # <repo-root>/
data_path = repo_root_path / "data" # <repo-root>/data
input_dataset_path = data_path / "anom_get_urls_counts.csv" # input: anom_get_urls_counts

In [3]:
anom_only_urls_file = repo_root_path / "data" / "anom_only_urls.paraquet"


anom_only_urls_df = pd.read_parquet(anom_only_urls_file)
print(f"loaded: {anom_only_urls_file}")
print(f"shape: {anom_only_urls_df.shape}")

loaded: /mnt/hgfs/workspace/mlhttp/data/anom_only_urls.paraquet
shape: (1615, 2)


In [4]:
anom_only_urls_df.head(30)

label,anom,norm
url,,
http://localhost:8080.BAK,19,0
http://localhost:8080.Bak,20,0
http://localhost:8080.INC,13,0
http://localhost:8080.Inc,25,0
http://localhost:8080.OLD,20,0
http://localhost:8080.Old,22,0
http://localhost:8080.bak,23,0
http://localhost:8080.inc,16,0
http://localhost:8080.java,16,0


**Observations:**  

Multiple Malformed short URLs variations:  
- With similar extensions `.BAK, .Bak, .bak` `.INC`, `.Inc`, `.inc` `.OLD`, `.Old`, `.old`, `java`, `.jsp`
- Cases where urls `http://localhost:8080` followed by
    -  (**No** `/`) direct `.` and extension
    -  `/` only
    -  `/` with `.` and extension
    - `/` and large numeric value
    - `/` and large numeric value and extension

Group these into two pattern families - 
- first three: (`/` or `.` or `/.`) and (<`extension`>)
- next two: <`large-numeric-id`>

In [5]:

# families = {
#     "outside_app":    ~anom_only_urls.str.startswith("http://localhost:8080/tienda1/"),
#     "backup_ext":      anom_only_urls.str.contains(r"\.(?:bak|old|inc|java)(?:\.|$)|~$|~\.", case=False, regex=True),
#     "numeric_id":     anom_only_urls.str.contains(r"\d{10,}"),
#     "asf_probe":      anom_only_urls.str.contains("asf-logo-wide", case=False),
#     "trailing_slash":      (anom_only_urls.str.endswith("/") & ~anom_only_urls.str.endswith("tienda1/")),
# }

Start with full set

In [6]:
# Original Dataset
anom_only_remaining_df = anom_only_urls_df
anom_only_remaining_df.shape

(1615, 2)

Define the pattern management and URL filtering utilities

In [7]:
# Pattern-family registry
pattern_families = {}

def new_pattern_family(family: str):
    """Create an empty pattern family (list of regex alternatives)."""
    pattern_families[family] = []


def extend_pattern(family: str, *patterns: str):
    """Add one or more regex alternatives to an existing family."""
    if family not in pattern_families:
        raise KeyError(f"family '{family}' not created — call new_pattern_family first")
    pattern_families[family].extend(patterns)   # extend - flat list of strings

def overwrite_pattern(family: str, *patterns: str):
    """Overwrite one or more regex alternatives to an existing family."""
    if family not in pattern_families:
        raise KeyError(f"family '{family}' not created — call new_pattern_family first")
     # Overwrite new given patterns
    pattern_families[family] = []
    pattern_families[family].extend(patterns)

# Pattern Compile
def compiled_pattern(families: dict) -> dict:
    """Join each family's alternatives into one regex string."""
    compiled = {name: "|".join(pats) for name, pats in families.items()}
    print(f"compiled_pattern: {compiled}")
    return compiled

# Filter dataset on compiled urls
def filter_urls(df_input, compiled):
    #urls = df_input.index.to_series()
    df_remaining = df_input
    for name, pat in compiled.items():
        print(f"{name}: {pat}")
        urls = df_remaining.index.to_series() 
        suffix_mask = urls.str.contains(pat, case=False, regex=True, na=False)

        
        df_suffix = df_remaining.loc[suffix_mask].copy()
        df_remaining = df_remaining.loc[~suffix_mask].copy()
        
        print(f"{name}_suffix URLs: {len(df_suffix)} URLs, {df_suffix['anom'].sum()} requests")
        print(f"remaining URLs:   {len(df_remaining)} URLs, {df_remaining['anom'].sum()} requests")
        print("-" * 80)
    return df_remaining

In [8]:
# extension followed pattern 
extension_pattern= r"\.(?:bak|old|inc|java|jsp)" 
numeric_id_pattern=(r"\d{10,}")

# Define a new extensions family and add pattern
new_pattern_family("extensions")
extend_pattern("extensions", extension_pattern)

# Define a new numeric-id family and add pattern
new_pattern_family("numeric-id")
extend_pattern("numeric-id", numeric_id_pattern)

# Compile the pattern
compiled = compiled_pattern(pattern_families)

# Filter URLs on compiled patterns
anom_only_remaining_df = filter_urls(anom_only_remaining_df, compiled)

compiled_pattern: {'extensions': '\\.(?:bak|old|inc|java|jsp)', 'numeric-id': '\\d{10,}'}
extensions: \.(?:bak|old|inc|java|jsp)
extensions_suffix URLs: 1308 URLs, 3398 requests
remaining URLs:   307 URLs, 2110 requests
--------------------------------------------------------------------------------
numeric-id: \d{10,}
numeric-id_suffix URLs: 26 URLs, 174 requests
remaining URLs:   281 URLs, 1936 requests
--------------------------------------------------------------------------------


In [9]:
print(f"Dataframe shape: {anom_only_remaining_df.shape}")
print(f"pattern_families:\n{json.dumps(pattern_families, indent=2)}")

Dataframe shape: (281, 2)
pattern_families:
{
  "extensions": [
    "\\.(?:bak|old|inc|java|jsp)"
  ],
  "numeric-id": [
    "\\d{10,}"
  ]
}


Sample next batch of URLs for analysis from the filtered dataset

In [10]:
anom_only_remaining_df.head(30)

label,anom,norm
url,,
http://localhost:8080/,51,0
http://localhost:8080/CFIDE/administrator/Application.cfm,5,0
http://localhost:8080/CFIDE/administrator/aboutcf.cfm,7,0
http://localhost:8080/CFIDE/administrator/checkfile.cfm,6,0
http://localhost:8080/CFIDE/administrator/enter.cfm,9,0
http://localhost:8080/CFIDE/administrator/header.cfm,9,0
http://localhost:8080/CFIDE/administrator/homefile.cfm,4,0
http://localhost:8080/CFIDE/administrator/homepage.cfm,8,0
http://localhost:8080/CFIDE/administrator/index.cfm,9,0


**Observations:**  

Multiple Malformed URLs variations:  
- Trailing slash `http://localhost:8080/` 	
- Adobe coldfusion scan probes - URLs with `CFIDE/...`
- IBM WebSphere scan probes - URLs with (`IBMWebAS`)
- MS IIS server scan probes - URLs with (`IISSamples/...`)

Group each one into its own respective pattern family 

In [11]:

# families = {
#     "outside_app":    ~anom_only_urls.str.startswith("http://localhost:8080/tienda1/"),
#     "backup_ext":      anom_only_urls.str.contains(r"\.(?:bak|old|inc|java)(?:\.|$)|~$|~\.", case=False, regex=True),
#     "numeric_id":     anom_only_urls.str.contains(r"\d{10,}"),
#     "asf_probe":      anom_only_urls.str.contains("asf-logo-wide", case=False),
#     "trailing_slash":      (anom_only_urls.str.endswith("/") & ~anom_only_urls.str.endswith("tienda1/")),
# }

In [12]:
# Define the patterns
trailing_slash_pattern = (r"^https?://[^/:]+:\d{1,5}/$")
adobe_coldfusion_patterns = (r"CFIDE/")
websphere_patterns = (r"IBMWebAS")
iis_patterns = (r"IISSamples")

# Define a new trailing_slash family and add pattern
new_pattern_family("trailing_slash")
extend_pattern("trailing_slash", trailing_slash_pattern)

# Define a new adobe coldfusion family and add pattern
new_pattern_family("adobe_coldfusion")
extend_pattern("adobe_coldfusion", adobe_coldfusion_patterns)

# Define a new websphere family and add pattern
new_pattern_family("websphere")
extend_pattern("websphere", websphere_patterns)

# Define a new websphere family and add pattern
new_pattern_family("iis")
extend_pattern("iis", iis_patterns)

# Compile the pattern
compiled = compiled_pattern(pattern_families)

# Reset to original Dataset
anom_only_remaining_df = anom_only_urls_df
print(anom_only_remaining_df.shape)

# Filter URLs on compiled patterns
anom_only_remaining_df = filter_urls(anom_only_remaining_df, compiled)

compiled_pattern: {'extensions': '\\.(?:bak|old|inc|java|jsp)', 'numeric-id': '\\d{10,}', 'trailing_slash': '^https?://[^/:]+:\\d{1,5}/$', 'adobe_coldfusion': 'CFIDE/', 'websphere': 'IBMWebAS', 'iis': 'IISSamples'}
(1615, 2)
extensions: \.(?:bak|old|inc|java|jsp)
extensions_suffix URLs: 1308 URLs, 3398 requests
remaining URLs:   307 URLs, 2110 requests
--------------------------------------------------------------------------------
numeric-id: \d{10,}
numeric-id_suffix URLs: 26 URLs, 174 requests
remaining URLs:   281 URLs, 1936 requests
--------------------------------------------------------------------------------
trailing_slash: ^https?://[^/:]+:\d{1,5}/$
trailing_slash_suffix URLs: 1 URLs, 51 requests
remaining URLs:   280 URLs, 1885 requests
--------------------------------------------------------------------------------
adobe_coldfusion: CFIDE/
adobe_coldfusion_suffix URLs: 20 URLs, 136 requests
remaining URLs:   260 URLs, 1749 requests
------------------------------------------

In [13]:
print(f"Dataframe shape: {anom_only_remaining_df.shape}")
print(f"pattern_families:\n{json.dumps(pattern_families, indent=2)}")

Dataframe shape: (238, 2)
pattern_families:
{
  "extensions": [
    "\\.(?:bak|old|inc|java|jsp)"
  ],
  "numeric-id": [
    "\\d{10,}"
  ],
  "trailing_slash": [
    "^https?://[^/:]+:\\d{1,5}/$"
  ],
  "adobe_coldfusion": [
    "CFIDE/"
  ],
  "websphere": [
    "IBMWebAS"
  ],
  "iis": [
    "IISSamples"
  ]
}


Sample next batch of URLs for analysis from the filtered dataset

In [14]:
anom_only_remaining_df.head(30)

label,anom,norm
url,,
http://localhost:8080/SmarTicketApp/index.html,6,0
http://localhost:8080/WEB-INF/web.xml,4,0
http://localhost:8080/WEB-INF/webapp.properties,3,0
http://localhost:8080/WebSphereSamples/,23,0
http://localhost:8080/WebSphereSamples/SingleSamples/AccountAndTransfer/create.html,8,0
http://localhost:8080/WebSphereSamples/SingleSamples/Increment/increment.html,11,0
http://localhost:8080/WebSphereSamples/YourCo/main.html,5,0
http://localhost:8080/_cti_pvt/access.cnf,3,0
http://localhost:8080/_cti_pvt/botinfs.cnf,8,0


**Observations:**  

Multiple Malformed URLs variations:  
- More variations of IBM WebSphere scan probes - URLs with (`webapp...`, `WebSphere...`, `SmarTicketApp...`)
- Microsoft frontpage scan probes - URLs with (`_cti...`, `_private...`, `_vti...`)
- Java metadata probes - URLs with `WEB-INF`

Extend the websphere pattern family and make new one for ms frontpage and Java app

In [15]:
# Define the patterns
more_websphere_patterns = (r"webapp|WebSphere|SmarTicketApp")
frontpage_patterns = (r"_cti|_private|_vti")
java_patterns = (r"WEB-INF")

# Extend a websphere family patterns
extend_pattern("websphere", more_websphere_patterns)

# Define a new frontpage family and add pattern
new_pattern_family("frontpage")
extend_pattern("frontpage", frontpage_patterns)

# Define a new java family and add pattern
new_pattern_family("java")
extend_pattern("java", java_patterns)

# Compile the pattern
compiled = compiled_pattern(pattern_families)

# Reset to original Dataset
anom_only_remaining_df = anom_only_urls_df
print(anom_only_remaining_df.shape)

# Filter URLs on compiled patterns
anom_only_remaining_df = filter_urls(anom_only_remaining_df, compiled)

compiled_pattern: {'extensions': '\\.(?:bak|old|inc|java|jsp)', 'numeric-id': '\\d{10,}', 'trailing_slash': '^https?://[^/:]+:\\d{1,5}/$', 'adobe_coldfusion': 'CFIDE/', 'websphere': 'IBMWebAS|webapp|WebSphere|SmarTicketApp', 'iis': 'IISSamples', 'frontpage': '_cti|_private|_vti', 'java': 'WEB-INF'}
(1615, 2)
extensions: \.(?:bak|old|inc|java|jsp)
extensions_suffix URLs: 1308 URLs, 3398 requests
remaining URLs:   307 URLs, 2110 requests
--------------------------------------------------------------------------------
numeric-id: \d{10,}
numeric-id_suffix URLs: 26 URLs, 174 requests
remaining URLs:   281 URLs, 1936 requests
--------------------------------------------------------------------------------
trailing_slash: ^https?://[^/:]+:\d{1,5}/$
trailing_slash_suffix URLs: 1 URLs, 51 requests
remaining URLs:   280 URLs, 1885 requests
--------------------------------------------------------------------------------
adobe_coldfusion: CFIDE/
adobe_coldfusion_suffix URLs: 20 URLs, 136 requests

In [16]:
print(f"Dataframe shape: {anom_only_remaining_df.shape}")
print(f"pattern_families:\n{json.dumps(pattern_families, indent=2)}")
bk_pattern_families = pattern_families

Dataframe shape: (200, 2)
pattern_families:
{
  "extensions": [
    "\\.(?:bak|old|inc|java|jsp)"
  ],
  "numeric-id": [
    "\\d{10,}"
  ],
  "trailing_slash": [
    "^https?://[^/:]+:\\d{1,5}/$"
  ],
  "adobe_coldfusion": [
    "CFIDE/"
  ],
  "websphere": [
    "IBMWebAS",
    "webapp|WebSphere|SmarTicketApp"
  ],
  "iis": [
    "IISSamples"
  ],
  "frontpage": [
    "_cti|_private|_vti"
  ],
  "java": [
    "WEB-INF"
  ]
}


In [17]:
pattern_families = bk_pattern_families
print(f"pattern_families:\n{json.dumps(pattern_families, indent=2)}")

pattern_families:
{
  "extensions": [
    "\\.(?:bak|old|inc|java|jsp)"
  ],
  "numeric-id": [
    "\\d{10,}"
  ],
  "trailing_slash": [
    "^https?://[^/:]+:\\d{1,5}/$"
  ],
  "adobe_coldfusion": [
    "CFIDE/"
  ],
  "websphere": [
    "IBMWebAS",
    "webapp|WebSphere|SmarTicketApp"
  ],
  "iis": [
    "IISSamples"
  ],
  "frontpage": [
    "_cti|_private|_vti"
  ],
  "java": [
    "WEB-INF"
  ]
}


Sample next batch of URLs for analysis from the filtered dataset

In [18]:
anom_only_remaining_df.head(30)

label,anom,norm
url,,
http://localhost:8080/admin.nsf,7,0
http://localhost:8080/admin/login.do,8,0
http://localhost:8080/admin4.nsf,11,0
http://localhost:8080/admin5.nsf,9,0
http://localhost:8080/agentrunner.nsf,4,0
http://localhost:8080/alog.nsf,6,0
http://localhost:8080/asf-logo-wide,39,0
http://localhost:8080/asf-logo-wide.gif,37,0
http://localhost:8080/asf-logo-wide.gif/,38,0


**Observations:**  

Multiple Malformed URLs variations: 
- Lotus Domino databases probes - - URLs with (`.nsf`)
- Apache Software Foundation probes - - URLs with (`asf-logo...`)
- More variations of IIS server scan probes - URLs with (`iisadmin...`, `iisadmpwd`)
- More variations of IBM WebSphere/J2EE scan probes - URLs with (`estore...`)
- WebLogic Portal probes - URLs with (`e2ePortalProject...`)
- Apache Struts application probe - URLs with (`admin/login.do`)
- Allaire/Macromedia JRun application probe - URLs with (`flash/java/javabean/FlashJavaBean...`)
- Known scanner wordlist probe - URLs with `databasenotes...`

In [19]:
# Define the patterns
lotus_domino_patterns = (r"\.nsf$")
asf_patterns = (r"asf-logo-wide")
more_iis_patterns = (r"iisadmin|iisadmpwd")
more_websphere_patterns = (r"estore")
portal_app_pattern = (r"e2ePortalProject")
# #more_java_patterns = (r"\.do$", r"flash/java/javabean|FlashJavaBean")
more_java_patterns = (r"\.do$|flash/java/javabean|FlashJavaBean")
file_patterns = (r"databasenotes\.html")

# Define a new lotus domino family and add pattern
new_pattern_family("lotus_domino")
extend_pattern("lotus_domino", lotus_domino_patterns)

# Define a new ASF family and add pattern
new_pattern_family("asf")
extend_pattern("asf", asf_patterns)

# Extend a IIS family patterns
extend_pattern("iis", more_iis_patterns)

# Extend a websphere family patterns
extend_pattern("websphere", more_websphere_patterns)

# Define a new frontpage family and add pattern
new_pattern_family("portal_app")
extend_pattern("portal_app", portal_app_pattern)

# Define a new java family and add pattern
extend_pattern("java", more_java_patterns)

# # Define a new file family and add pattern
new_pattern_family("file")
extend_pattern("file", file_patterns)

# Compile the pattern
compiled = compiled_pattern(pattern_families)

# Reset to original Dataset
anom_only_remaining_df = anom_only_urls_df
print(anom_only_remaining_df.shape)

# Filter URLs on compiled patterns
anom_only_remaining_df = filter_urls(anom_only_remaining_df, compiled)

compiled_pattern: {'extensions': '\\.(?:bak|old|inc|java|jsp)', 'numeric-id': '\\d{10,}', 'trailing_slash': '^https?://[^/:]+:\\d{1,5}/$', 'adobe_coldfusion': 'CFIDE/', 'websphere': 'IBMWebAS|webapp|WebSphere|SmarTicketApp|estore', 'iis': 'IISSamples|iisadmin|iisadmpwd', 'frontpage': '_cti|_private|_vti', 'java': 'WEB-INF|\\.do$|flash/java/javabean|FlashJavaBean', 'lotus_domino': '\\.nsf$', 'asf': 'asf-logo-wide', 'portal_app': 'e2ePortalProject', 'file': 'databasenotes\\.html'}
(1615, 2)
extensions: \.(?:bak|old|inc|java|jsp)
extensions_suffix URLs: 1308 URLs, 3398 requests
remaining URLs:   307 URLs, 2110 requests
--------------------------------------------------------------------------------
numeric-id: \d{10,}
numeric-id_suffix URLs: 26 URLs, 174 requests
remaining URLs:   281 URLs, 1936 requests
--------------------------------------------------------------------------------
trailing_slash: ^https?://[^/:]+:\d{1,5}/$
trailing_slash_suffix URLs: 1 URLs, 51 requests
remaining URLs:

In [20]:
print(f"Dataframe shape: {anom_only_remaining_df.shape}")
print(f"pattern_families:\n{json.dumps(pattern_families, indent=2)}")

Dataframe shape: (91, 2)
pattern_families:
{
  "extensions": [
    "\\.(?:bak|old|inc|java|jsp)"
  ],
  "numeric-id": [
    "\\d{10,}"
  ],
  "trailing_slash": [
    "^https?://[^/:]+:\\d{1,5}/$"
  ],
  "adobe_coldfusion": [
    "CFIDE/"
  ],
  "websphere": [
    "IBMWebAS",
    "webapp|WebSphere|SmarTicketApp",
    "estore"
  ],
  "iis": [
    "IISSamples",
    "iisadmin|iisadmpwd"
  ],
  "frontpage": [
    "_cti|_private|_vti"
  ],
  "java": [
    "WEB-INF",
    "\\.do$|flash/java/javabean|FlashJavaBean"
  ],
  "lotus_domino": [
    "\\.nsf$"
  ],
  "asf": [
    "asf-logo-wide"
  ],
  "portal_app": [
    "e2ePortalProject"
  ],
  "file": [
    "databasenotes\\.html"
  ]
}


Sample next batch of URLs for analysis from the filtered dataset

In [21]:
anom_only_remaining_df.head(30)

label,anom,norm
url,,
http://localhost:8080/iisstart.asp,4,0
http://localhost:8080/iisstart.htm,9,0
http://localhost:8080/jstl-war/index.html,9,0
http://localhost:8080/mail.box,6,0
http://localhost:8080/msadc/msadcs.dll,16,0
http://localhost:8080/msadc/readme.txt,2,0
http://localhost:8080/msadc/samples/adctest.asp,6,0
http://localhost:8080/msadc/samples/selector/showcode.asp,8,0
http://localhost:8080/postinfo.html,5,0


**Observations:**  

Multiple Malformed URLs variations: 
- More variations of IIS scan probes - URLs with (`iisstart...`, `msadc...`, `samples...`)
- More variations Java metadata probes - URLs with `jstl-war...`
- More Lotus Domino databases probes - - URLs with (`mail.box`)
- More Microsoft frontpage scan probes - URLs with (`postinfo...`)  

In [22]:
# Define the patterns
more_iis_patterns = (r"iisstart|msadc")
some_more_iis_patterns = (r"/samples/(?:dbsamp|gbook|htmlsamp|isapi|sampsite)/")
more_java_patterns = (r"jstl-war")
more_lotus_domino_patterns = (r"/mail\.box$")
more_frontpage_patterns = (r"postinfo\.html")

# Add more IIS family patterns
extend_pattern("iis", more_iis_patterns)
extend_pattern("iis", some_more_iis_patterns)

# Add more Java family patterns
extend_pattern("java", more_java_patterns)

# Add more  domino family patterns
extend_pattern("lotus_domino", more_lotus_domino_patterns)

# Add more frontpage family patterns
extend_pattern("frontpage", more_frontpage_patterns)

# Compile the pattern
compiled = compiled_pattern(pattern_families)

# Reset to original Dataset
anom_only_remaining_df = anom_only_urls_df
print(anom_only_remaining_df.shape)

# Filter URLs on compiled patterns
anom_only_remaining_df = filter_urls(anom_only_remaining_df, compiled)

compiled_pattern: {'extensions': '\\.(?:bak|old|inc|java|jsp)', 'numeric-id': '\\d{10,}', 'trailing_slash': '^https?://[^/:]+:\\d{1,5}/$', 'adobe_coldfusion': 'CFIDE/', 'websphere': 'IBMWebAS|webapp|WebSphere|SmarTicketApp|estore', 'iis': 'IISSamples|iisadmin|iisadmpwd|iisstart|msadc|/samples/(?:dbsamp|gbook|htmlsamp|isapi|sampsite)/', 'frontpage': '_cti|_private|_vti|postinfo\\.html', 'java': 'WEB-INF|\\.do$|flash/java/javabean|FlashJavaBean|jstl-war', 'lotus_domino': '\\.nsf$|/mail\\.box$', 'asf': 'asf-logo-wide', 'portal_app': 'e2ePortalProject', 'file': 'databasenotes\\.html'}
(1615, 2)
extensions: \.(?:bak|old|inc|java|jsp)
extensions_suffix URLs: 1308 URLs, 3398 requests
remaining URLs:   307 URLs, 2110 requests
--------------------------------------------------------------------------------
numeric-id: \d{10,}
numeric-id_suffix URLs: 26 URLs, 174 requests
remaining URLs:   281 URLs, 1936 requests
--------------------------------------------------------------------------------
tr

In [23]:
print(f"Dataframe shape: {anom_only_remaining_df.shape}")
print(f"pattern_families:\n{json.dumps(pattern_families, indent=2)}")

Dataframe shape: (60, 2)
pattern_families:
{
  "extensions": [
    "\\.(?:bak|old|inc|java|jsp)"
  ],
  "numeric-id": [
    "\\d{10,}"
  ],
  "trailing_slash": [
    "^https?://[^/:]+:\\d{1,5}/$"
  ],
  "adobe_coldfusion": [
    "CFIDE/"
  ],
  "websphere": [
    "IBMWebAS",
    "webapp|WebSphere|SmarTicketApp",
    "estore"
  ],
  "iis": [
    "IISSamples",
    "iisadmin|iisadmpwd",
    "iisstart|msadc",
    "/samples/(?:dbsamp|gbook|htmlsamp|isapi|sampsite)/"
  ],
  "frontpage": [
    "_cti|_private|_vti",
    "postinfo\\.html"
  ],
  "java": [
    "WEB-INF",
    "\\.do$|flash/java/javabean|FlashJavaBean",
    "jstl-war"
  ],
  "lotus_domino": [
    "\\.nsf$",
    "/mail\\.box$"
  ],
  "asf": [
    "asf-logo-wide"
  ],
  "portal_app": [
    "e2ePortalProject"
  ],
  "file": [
    "databasenotes\\.html"
  ]
}


---

Sample next batch of URLs for analysis from the filtered dataset

In [24]:
anom_only_remaining_df.head(30)

label,anom,norm
url,,
http://localhost:8080/scripts/tools/drvrs.exe,14,0
http://localhost:8080/scripts/tools/dsnform.exe,4,0
http://localhost:8080/scripts/tools/mkilog.exe,9,0
http://localhost:8080/scripts/tools/newdsn.exe,7,0
http://localhost:8080/servlet/ErrorReporter,5,0
http://localhost:8080/servlet/HelloWorldServlet,8,0
http://localhost:8080/servlet/SimpleServlet,8,0
http://localhost:8080/servlet/SnoopServlet,9,0
http://localhost:8080/servlet/aphtpassword,5,0


**Observations:**  

Multiple Malformed URLs variations: 
- More variations of IIS scan probes - URLs with (`/scripts/tools...`)
- More variations of Java metadata probes - URLs with `servlet...`)
- More variations of websphere probes - URLs with `techniques/servlets...`)
- Wordlist (generic bait) probe - URLs with (`theme`)
- Editor backup file suffix `~` *(emacs/vi convention)* - URLs with (`tienda1/...~`)
- URL ends with trailing slash - change earlier defined `trailing_slash_pattern` `(r"^https?://[^/:]+:\d{1,5}/$")` to cover for URLs api/endpoint/paths/ that ends with `/`

In [26]:
# Define the patterns
more_iis_patterns = (r"/scripts/tools/.*\.exe")
more_java_patterns = (r"/servlet/(?:Snoop|Hello|Simple|snoop2?|ErrorReporter|aphtpassword|hello)")
more_websphere_patterns = (r"techniques/servlets")
#known_scanner_bait_pattern = (r"^http://localhost:8080/theme$")
known_scanner_bait_pattern = (r"^https?://[^/:]+:\d{1,5}/theme$")
tilde_backup_file_pattern = (r"~$|~/|/~$")
overwrite_trailing_slash_pattern = (r"^https?://[^/:]+:\d{1,5}.*/$")

# Add more IIS family patterns
extend_pattern("iis", more_iis_patterns)

# Add more Java family patterns
extend_pattern("java", more_java_patterns)

# Add more websphere family patterns
extend_pattern("websphere", more_websphere_patterns)

# Add new scanner bait family and add patterns
new_pattern_family("known_scanner_bait")
extend_pattern("known_scanner_bait", known_scanner_bait_pattern)

# Add new tilde family and add patterns
new_pattern_family("tilde_backup_file")
extend_pattern("tilde_backup_file", tilde_backup_file_pattern)

# Overwrite trailing slash family and add patterns
overwrite_pattern("trailing_slash", overwrite_trailing_slash_pattern)

# Compile the pattern
compiled = compiled_pattern(pattern_families)

# Reset to original Dataset
anom_only_remaining_df = anom_only_urls_df
print(anom_only_remaining_df.shape)

# Filter URLs on compiled patterns
anom_only_remaining_df = filter_urls(anom_only_remaining_df, compiled)

compiled_pattern: {'extensions': '\\.(?:bak|old|inc|java|jsp)', 'numeric-id': '\\d{10,}', 'trailing_slash': '^https?://[^/:]+:\\d{1,5}.*/$', 'adobe_coldfusion': 'CFIDE/', 'websphere': 'IBMWebAS|webapp|WebSphere|SmarTicketApp|estore|techniques/servlets|techniques/servlets', 'iis': 'IISSamples|iisadmin|iisadmpwd|iisstart|msadc|/samples/(?:dbsamp|gbook|htmlsamp|isapi|sampsite)/|/scripts/tools/.*\\.exe|/scripts/tools/.*\\.exe', 'frontpage': '_cti|_private|_vti|postinfo\\.html', 'java': 'WEB-INF|\\.do$|flash/java/javabean|FlashJavaBean|jstl-war|/servlet/(?:Snoop|Hello|Simple|snoop2?|ErrorReporter|aphtpassword|hello)|/servlet/(?:Snoop|Hello|Simple|snoop2?|ErrorReporter|aphtpassword|hello)', 'lotus_domino': '\\.nsf$|/mail\\.box$', 'asf': 'asf-logo-wide', 'portal_app': 'e2ePortalProject', 'file': 'databasenotes\\.html', 'known_scanner_bait': '^https?://[^/:]+:\\d{1,5}/theme$', 'tilde_backup_file': '~$|~/|/~$'}
(1615, 2)
extensions: \.(?:bak|old|inc|java|jsp)
extensions_suffix URLs: 1308 URLs, 

In [27]:
print(f"Dataframe shape: {anom_only_remaining_df.shape}")
print(f"pattern_families:\n{json.dumps(pattern_families, indent=2)}")

Dataframe shape: (3, 2)
pattern_families:
{
  "extensions": [
    "\\.(?:bak|old|inc|java|jsp)"
  ],
  "numeric-id": [
    "\\d{10,}"
  ],
  "trailing_slash": [
    "^https?://[^/:]+:\\d{1,5}.*/$"
  ],
  "adobe_coldfusion": [
    "CFIDE/"
  ],
  "websphere": [
    "IBMWebAS",
    "webapp|WebSphere|SmarTicketApp",
    "estore",
    "techniques/servlets",
    "techniques/servlets"
  ],
  "iis": [
    "IISSamples",
    "iisadmin|iisadmpwd",
    "iisstart|msadc",
    "/samples/(?:dbsamp|gbook|htmlsamp|isapi|sampsite)/",
    "/scripts/tools/.*\\.exe",
    "/scripts/tools/.*\\.exe"
  ],
  "frontpage": [
    "_cti|_private|_vti",
    "postinfo\\.html"
  ],
  "java": [
    "WEB-INF",
    "\\.do$|flash/java/javabean|FlashJavaBean",
    "jstl-war",
    "/servlet/(?:Snoop|Hello|Simple|snoop2?|ErrorReporter|aphtpassword|hello)",
    "/servlet/(?:Snoop|Hello|Simple|snoop2?|ErrorReporter|aphtpassword|hello)"
  ],
  "lotus_domino": [
    "\\.nsf$",
    "/mail\\.box$"
  ],
  "asf": [
    "asf-logo-wid

---

Sample next batch of URLs for analysis from the filtered dataset

In [28]:
anom_only_remaining_df.head(30)

label,anom,norm
url,,
http://localhost:8080/wliconsole,5,0
http://localhost:8080/worldmusic/action/catalog,5,0
http://localhost:8080/worldmusic/action/cdlist,4,0


In [ ]:
**Observations:**  

Last few URL variations: 
- More variations of WebSphere scan probes - URLs with (`wliconsole`)
- More variations of Java probes - URLs with `worldmusic...`)

In [29]:
# Define the patterns
more_java_patterns = (r"wliconsole")
more_websphere_patterns = (r"worldmusic")

# Add more Java family patterns
extend_pattern("java", more_java_patterns)

# Add more websphere family patterns
extend_pattern("websphere", more_websphere_patterns)

# Compile the pattern
compiled = compiled_pattern(pattern_families)

# Reset to original Dataset
anom_only_remaining_df = anom_only_urls_df
print(anom_only_remaining_df.shape)

# Filter URLs on compiled patterns
anom_only_remaining_df = filter_urls(anom_only_remaining_df, compiled)

compiled_pattern: {'extensions': '\\.(?:bak|old|inc|java|jsp)', 'numeric-id': '\\d{10,}', 'trailing_slash': '^https?://[^/:]+:\\d{1,5}.*/$', 'adobe_coldfusion': 'CFIDE/', 'websphere': 'IBMWebAS|webapp|WebSphere|SmarTicketApp|estore|techniques/servlets|techniques/servlets|worldmusic', 'iis': 'IISSamples|iisadmin|iisadmpwd|iisstart|msadc|/samples/(?:dbsamp|gbook|htmlsamp|isapi|sampsite)/|/scripts/tools/.*\\.exe|/scripts/tools/.*\\.exe', 'frontpage': '_cti|_private|_vti|postinfo\\.html', 'java': 'WEB-INF|\\.do$|flash/java/javabean|FlashJavaBean|jstl-war|/servlet/(?:Snoop|Hello|Simple|snoop2?|ErrorReporter|aphtpassword|hello)|/servlet/(?:Snoop|Hello|Simple|snoop2?|ErrorReporter|aphtpassword|hello)|wliconsole', 'lotus_domino': '\\.nsf$|/mail\\.box$', 'asf': 'asf-logo-wide', 'portal_app': 'e2ePortalProject', 'file': 'databasenotes\\.html', 'known_scanner_bait': '^https?://[^/:]+:\\d{1,5}/theme$', 'tilde_backup_file': '~$|~/|/~$'}
(1615, 2)
extensions: \.(?:bak|old|inc|java|jsp)
extensions_su

In [31]:
print(f"Dataframe shape: {anom_only_remaining_df.shape}")
print(f"pattern_families:\n{json.dumps(pattern_families, indent=2)}")

Dataframe shape: (0, 2)
pattern_families:
{
  "extensions": [
    "\\.(?:bak|old|inc|java|jsp)"
  ],
  "numeric-id": [
    "\\d{10,}"
  ],
  "trailing_slash": [
    "^https?://[^/:]+:\\d{1,5}.*/$"
  ],
  "adobe_coldfusion": [
    "CFIDE/"
  ],
  "websphere": [
    "IBMWebAS",
    "webapp|WebSphere|SmarTicketApp",
    "estore",
    "techniques/servlets",
    "techniques/servlets",
    "worldmusic"
  ],
  "iis": [
    "IISSamples",
    "iisadmin|iisadmpwd",
    "iisstart|msadc",
    "/samples/(?:dbsamp|gbook|htmlsamp|isapi|sampsite)/",
    "/scripts/tools/.*\\.exe",
    "/scripts/tools/.*\\.exe"
  ],
  "frontpage": [
    "_cti|_private|_vti",
    "postinfo\\.html"
  ],
  "java": [
    "WEB-INF",
    "\\.do$|flash/java/javabean|FlashJavaBean",
    "jstl-war",
    "/servlet/(?:Snoop|Hello|Simple|snoop2?|ErrorReporter|aphtpassword|hello)",
    "/servlet/(?:Snoop|Hello|Simple|snoop2?|ErrorReporter|aphtpassword|hello)",
    "wliconsole"
  ],
  "lotus_domino": [
    "\\.nsf$",
    "/mail\\.box

The anomalous URLs are now fully bucketed into families. These can be used for feature engineering

---